# Equipo 19 · Avance 4: variaciones del entrenamiento con la máscara de Arnold et al. (2010)

La [prueba con Arnold del Avance 3](Prueba_Entrenamiento_con_Arnold_Ejecutado.ipynb) usó lr 1e-4 y validó con
`c2_transverse1`. Las [corridas 1 a 5 con El Meslouhi](../meslouhi/) mostraron después que la mejor configuración es
la de la **corrida 4**: lr 1e-5, lote 4, 2000 iteraciones, con `c1_ascending` en entrenamiento y `c2_sigmoid` en
validación. Este notebook entrena Endo-STTN con la máscara de Arnold usando esa configuración, en tres variantes:

| Variante | Cambio | Pregunta |
|---|---|---|
| **A1** | Configuración de la corrida 4 (semilla 2020) | Con la mejor configuración, ¿Arnold rinde como El Meslouhi? |
| **A2** | Igual que A1, semilla 2021 | ¿El resultado se repite? |
| **A3** | Igual que A1, hasta 4000 iteraciones (continúa desde A1) | ¿Entrenar más ayuda con lr bajo? |

**Una variante por sesión de Colab:** elige `VARIANTE` en la celda de parámetros y corre todas las celdas. Corre A1
antes que A3: A3 copia los checkpoints de A1 y sigue desde la iteración 2000, así no repite lo ya entrenado.

**Comparación justa con El Meslouhi.** El hueco desplazado (*pseudo ground truth*) sale de la propia máscara de
reflejos, así que con Arnold los huecos de prueba son otros y las métricas no se comparan directamente con las de
las corridas 1 a 5 (el mismo modelo original da 29.09 dB con huecos de Arnold y 29.51 dB con los de El Meslouhi).
Por eso cada variante se evalúa **dos veces**:
1. con los huecos de **Arnold** (la máscara con la que se entrenó), y
2. con los huecos de **El Meslouhi** (`evaluar_endosttn.py --mascara-eval meslouhi`), los mismos de las corridas 1 a 5.

La segunda es la que permite decir si Arnold entrena un Endo-STTN mejor o peor que El Meslouhi.

## 0. Preparar Colab

Activa la GPU (*Entorno de ejecución → Cambiar tipo de entorno de ejecución*; mejor A100 o L4, T4 también sirve con `LOTE = 2`).
Usa los ZIPs de C3VDv2 de `MyDrive/Equipo19/raw/`. Cada variante se guarda en
`MyDrive/Equipo19/re_entrenar_endo/arnold_variaciones/<VARIANTE>/`, aparte de las corridas anteriores.
**Si Colab se desconecta durante el entrenamiento, vuelve a correr todas las celdas: continúa desde el último checkpoint.**

In [ ]:
%matplotlib inline
from pathlib import Path
import os, sys, json, shutil, subprocess, zipfile

# ---------- Ajusta aquí ----------
VARIANTE = "A1"        # "A1", "A2" o "A3"
LOTE = 4               # clips de 5 frames por lote; bájalo a 2 si falta memoria de GPU (T4)
CORRER_PREPARAR = True
CORRER_ENTRENAR = True
CORRER_EVALUAR  = True
DRIVE = Path("/content/drive/MyDrive/Equipo19")
REPO_URL = "https://github.com/Jkalks2025/Eliminaci-n-de-artefactos-en-im-genes-de-endoscop-a.git"
RAMA = "claude/beautiful-einstein-pd9k6x"   # la que trae evaluar_endosttn.py --mascara-eval
# ---------------------------------

VARIANTES = {   # lo común: máscara arnold, lr 1e-5, lote 4, un checkpoint cada 250 iteraciones
    "A1": {"semilla": 2020, "iteraciones": 2000, "continuar_de": None},
    "A2": {"semilla": 2021, "iteraciones": 2000, "continuar_de": None},
    "A3": {"semilla": 2020, "iteraciones": 4000, "continuar_de": "A1"},
}
CFG = VARIANTES[VARIANTE]
MASCARA = "arnold"
LR = 1e-5
GUARDAR_CADA = 250
PASO = 1
DIVISION = {   # la de la corrida 5 con El Meslouhi (si el ZIP trae sufijo, p. ej. "c1_ascending_t3_v1-007", se reconoce solo)
    "entrenamiento": ["c2_cecum_t1_v1", "c2_rectum_t1_v1", "c2_transverse1_t1_v1", "c1_ascending_t3_v1"],
    "validacion": ["c2_sigmoid_t3_v2"],
    "prueba": ["c2_descending_t4_v2", "c2_descending_t4_v3"],
}

assert "google.colab" in sys.modules, "Este notebook es para Google Colab"
from google.colab import drive
drive.mount("/content/drive")
BASE = DRIVE / "re_entrenar_endo" / "arnold_variaciones"
GUARDADO = BASE / VARIANTE
GUARDADO.mkdir(parents=True, exist_ok=True)

REPO = Path("/content/repo")
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "-b", RAMA, REPO_URL, str(REPO)], check=True)
subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin", RAMA], check=True)
subprocess.run(["git", "-C", str(REPO), "checkout", "-q", "-B", RAMA, f"origin/{RAMA}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown", "tabulate", "tensorboardX"], check=True)
pre = REPO / "Endo-STTN" / "release_model" / "pretrained_model"
if (DRIVE / "gen_00009.pth").exists() and not (pre / "gen_00009.pth").exists():
    pre.mkdir(parents=True, exist_ok=True); shutil.copy(DRIVE / "gen_00009.pth", pre)
subprocess.run([sys.executable, str(REPO / "preparar_entorno.py")], check=True)

datos = REPO / "datos"
assert (DRIVE / "raw").exists(), f"No encuentro {DRIVE / 'raw'}: ahí deben estar los ZIPs de C3VDv2"
for z in (DRIVE / "raw").glob("*.zip"):                    # se leen directo de Drive, sin copiarlos
    if not (datos / "raw" / z.name).exists():
        os.symlink(z, datos / "raw" / z.name)
def nombre_zip(video):
    """Nombre del ZIP en Drive/raw para un video: exacto, o con sufijo de Drive ("-007", " (1)")."""
    stems = sorted(z.stem for z in (DRIVE / "raw").glob("*.zip"))
    if video in stems:
        return video
    candidatos = [s for s in stems if s.startswith(video) and not s[len(video)].isalnum() and s[len(video)] != "_"]
    assert len(candidatos) == 1, (f"Para {video} hay {len(candidatos)} ZIPs en Drive/raw: {candidatos or 'ninguno'}. "
                                  f"ZIPs disponibles: {stems}")
    return candidatos[0]

DIVISION = {g: [nombre_zip(v) for v in vs] for g, vs in DIVISION.items()}
print("División (nombres de los ZIPs):", json.dumps(DIVISION, indent=1))
prep_local = datos / "endosttn_c3vd" / "preparacion.json"
if prep_local.exists() and (json.load(open(prep_local)).get("mascara"), json.load(open(prep_local))["division"]) != (MASCARA, DIVISION):
    shutil.rmtree(datos / "endosttn_c3vd")                  # preparado con otra máscara o división en esta sesión
if (BASE / "endosttn_c3vd.zip").exists() and not prep_local.exists():
    shutil.unpack_archive(str(BASE / "endosttn_c3vd.zip"), str(datos))   # mismo dataset para las tres variantes

modelos_drive = GUARDADO / "modelos"
if CFG["continuar_de"] and not modelos_drive.exists():     # A3: copia los checkpoints de A1 y sigue desde ahí
    origen = BASE / CFG["continuar_de"] / "modelos"
    assert (origen / "latest.ckpt").exists(), f"Corre primero {CFG['continuar_de']}: no encuentro {origen}"
    shutil.copytree(origen, modelos_drive, ignore=shutil.ignore_patterns("inicio", "gen_mejor.pth"))
    print(f"Checkpoints de {CFG['continuar_de']} copiados a {modelos_drive}")
modelos_drive.mkdir(exist_ok=True)
(REPO / "modelos").mkdir(exist_ok=True)
enlace = REPO / "modelos" / "endosttn_c3vd"                 # checkpoints directo en Drive
if enlace.is_symlink() and Path(os.readlink(enlace)) != modelos_drive:
    enlace.unlink()
if not enlace.exists():
    os.symlink(modelos_drive, enlace)
RES_ARNOLD = GUARDADO / "resultados" / "huecos_arnold"
RES_MESLOUHI = GUARDADO / "resultados" / "huecos_meslouhi"
os.environ["EQUIPO19_RESULTADOS_31"] = str(RES_ARNOLD)
PESOS_ORIGINALES = pre / "gen_00009.pth"
print(f"Variante {VARIANTE}: {CFG} | lr {LR} | lote {LOTE} | máscara {MASCARA}")
print("Todo se guarda en", GUARDADO)

In [ ]:
os.chdir(REPO)
A31 = REPO / "Avance3_ReEntrenar_Endo"
sys.path.insert(0, str(A31)); sys.path.insert(0, str(REPO / "codigo"))
import numpy as np, pandas as pd, torch, matplotlib.pyplot as plt
from PIL import Image
from IPython.display import Markdown, display
import endosttn_c3vd as ec
from endosttn_c3vd import c3vd

def correr(*args, resultados=RES_ARNOLD):
    """Corre un script y muestra su salida mientras avanza; sus resultados van a la carpeta indicada."""
    env = dict(os.environ, PYTHONWARNINGS="ignore", EQUIPO19_RESULTADOS_31=str(resultados))
    p = subprocess.Popen([sys.executable, "-u", *map(str, args)], stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, env=env)
    for linea in p.stdout:
        print(linea, end="", flush=True)
    assert p.wait() == 0, f"falló: {args}"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo :", DEVICE, torch.cuda.get_device_name(0) if DEVICE == "cuda" else "(sin GPU: entrenar será muy lento)")
print("Endo-STTN   :", PESOS_ORIGINALES, "OK" if PESOS_ORIGINALES.exists() else "NO ENCONTRADO")
faltan = [v for g in DIVISION.values() for v in g if not (c3vd.DIR_RAW / f"{v}.zip").exists()]
assert not faltan, f"Faltan estos videos en Drive/raw: {faltan}"

## 1. Datos y máscaras

Mismo dataset para las tres variantes (máscara de Arnold, división de la corrida 5). La primera variante que se corre
lo prepara y lo guarda en `arnold_variaciones/endosttn_c3vd.zip`; las siguientes lo reutilizan.

In [ ]:
if CORRER_PREPARAR:
    json.dump(DIVISION, open(BASE / "division.json", "w"), indent=2)
    correr(A31 / "preparar_endosttn.py", "--paso", PASO, "--mascara", MASCARA, "--division", BASE / "division.json")
    if not (BASE / "endosttn_c3vd.zip").exists():
        shutil.make_archive(str(BASE / "endosttn_c3vd"), "zip", root_dir=REPO / "datos", base_dir="endosttn_c3vd")

prep = json.load(open(ec.DIR_DATASET / "preparacion.json"))
display(pd.DataFrame(prep["videos"]).T)

In [ ]:
# Arnold vs El Meslouhi en 4 frames de entrenamiento al azar (magenta = reflejo detectado)
trozos = json.load(open(ec.DIR_DATASET / "train.json"))
rng = np.random.default_rng(0)
muestra = [(t, int(rng.integers(n))) for t, n in (list(trozos.items())[k] for k in rng.integers(len(trozos), size=4))]
color = np.array([255, 0, 255], np.uint8)
con_mascara = lambda img, m: np.where(m[..., None], (0.35 * img + 0.65 * color).astype(np.uint8), img)

fig, ax = plt.subplots(3, 4, figsize=(15, 11.5))
for j, (t, i) in enumerate(muestra):
    f = np.array(Image.open(zipfile.ZipFile(ec.DIR_DATASET / "JPEGImages" / f"{t}.zip").open(f"{i:05d}.png")))
    m_a, m_m = ec.mascara_arnold(f), ec.mascara_meslouhi(f)
    for r, (img, tit) in enumerate([(f, t), (con_mascara(f, m_a), f"arnold · {100 * m_a.mean():.1f}%"),
                                    (con_mascara(f, m_m), f"meslouhi · {100 * m_m.mean():.1f}%")]):
        ax[r, j].imshow(img); ax[r, j].set_title(tit, fontsize=10); ax[r, j].axis("off")
plt.suptitle("Arnold vs El Meslouhi (magenta = reflejo detectado)", y=1.0); plt.tight_layout()
RES_ARNOLD.mkdir(parents=True, exist_ok=True)
plt.savefig(RES_ARNOLD / "revision_mascaras.png", dpi=80, bbox_inches="tight"); plt.show()

## 2. Entrenamiento

Entrenamiento original de Endo-STTN desde `gen_00009.pth`, con **lr 1e-5** (en la prueba del Avance 3 era 1e-4) y la
semilla de la variante. Un checkpoint cada 250 iteraciones, como en las corridas 3 a 5. En A3, la salida debe decir
*Continuando desde la iteración 2000*; si dice que empieza de cero, revisa que A1 haya terminado.

In [ ]:
if CORRER_ENTRENAR:
    correr(A31 / "entrenar_endosttn.py", "--pesos", PESOS_ORIGINALES, "--iteraciones", CFG["iteraciones"],
           "--guardar-cada", GUARDAR_CADA, "--lote", LOTE, "--lr", LR, "--semilla", CFG["semilla"])
print(sorted(p.name for p in ec.DIR_ENTRENAMIENTO.glob("gen_[0-9]*.pth")))

## 3. Evaluación

Dos veces, con los mismos checkpoints y los mismos videos de prueba (`c2_descending_t4_v2` y `v3`):
1. **Huecos de Arnold:** elige `gen_mejor.pth` en validación y compara contra el original.
2. **Huecos de El Meslouhi:** los de las corridas 1 a 5. Comprobación: el **original** debe dar aquí lo mismo que en
   esas corridas (PSNR 29.51 dB, SSIM 0.826, MAE 6.66). Si no, los huecos no son los mismos y la comparación no vale.

In [ ]:
if CORRER_EVALUAR:
    correr(A31 / "evaluar_endosttn.py", "--pesos-originales", PESOS_ORIGINALES, resultados=RES_ARNOLD)
    correr(A31 / "evaluar_endosttn.py", "--pesos-originales", PESOS_ORIGINALES, "--mascara-eval", "meslouhi",
           resultados=RES_MESLOUHI)
for nombre, d in [("huecos de Arnold", RES_ARNOLD), ("huecos de El Meslouhi", RES_MESLOUHI)]:
    display(Markdown(f"### Evaluación con {nombre}\n\n" + open(d / "resumen.md").read()))

In [ ]:
# Resumen de la variante y referencia: corrida 4 con El Meslouhi (la mejor), evaluada con huecos de El Meslouhi
filas = []
for huecos, d in [("arnold", RES_ARNOLD), ("meslouhi", RES_MESLOUHI)]:
    t = pd.read_csv(d / "prueba_por_frame.csv").groupby("modelo")[["psnr", "ssim", "mae"]].mean()
    for modelo, r in t.iterrows():
        filas.append({"huecos": huecos, "modelo": f"{modelo} ({VARIANTE})" if modelo == "reentrenado" else modelo,
                      **r.round(4).to_dict()})
filas.append({"huecos": "meslouhi", "modelo": "reentrenado (corrida 4, El Meslouhi)", "psnr": 32.5234, "ssim": 0.8786, "mae": 4.2971})
tabla = pd.DataFrame(filas)
tabla.to_csv(GUARDADO / "resultados" / "comparacion.csv", index=False)
orig = tabla[(tabla.huecos == "meslouhi") & (tabla.modelo == "original")].iloc[0]
print("Comprobación del original con huecos de El Meslouhi:", "OK" if abs(orig.psnr - 29.5098) < 0.01 else
      f"NO coincide ({orig.psnr:.4f} dB en lugar de 29.5098): los huecos o los videos de prueba no son los mismos")
display(tabla)

In [ ]:
# Validación por checkpoint con los dos tipos de huecos
fig, ax = plt.subplots(1, 2, figsize=(14, 3.8))
for a, (huecos, d) in zip(ax, [("Arnold", RES_ARNOLD), ("El Meslouhi", RES_MESLOUHI)]):
    val = pd.read_csv(d / "validacion_por_checkpoint.csv")
    orig, ck = val[val.modelo == "original"].iloc[0], val[val.modelo != "original"]
    a.plot(ck.iteracion, ck.mae, "-o", color="#eb6834", lw=2, ms=7, label=f"reentrenado {VARIANTE}")
    a.axhline(orig.mae, color="#2a78d6", lw=2, ls="--", label="original (gen_00009)")
    a.set_title(f"Validación (c2_sigmoid) · huecos de {huecos}"); a.set_xlabel("iteración")
    a.set_ylabel("MAE en el hueco"); a.grid(alpha=0.3); a.legend()
plt.tight_layout(); plt.savefig(GUARDADO / "resultados" / "curvas_validacion.png", dpi=90); plt.show()

## 4. Qué subir a GitHub

Copia `MyDrive/Equipo19/re_entrenar_endo/arnold_variaciones/<VARIANTE>/resultados/` completa a
`Avance4_ComparacionMascaras/resultados/arnold/<VARIANTE>/`, junto con este notebook ejecutado
(`Arnold_variaciones_<VARIANTE>_Ejecutado.ipynb`), y `modelos/config.json` y `modelos/mejor.json`.
No subas los `.pth` (66 MB cada uno).

En A3, `config.json` es el de A1 (el entrenamiento se continuó): dice 2000 iteraciones y la fecha de A1.

In [ ]:
cfg = json.load(open(ec.DIR_ENTRENAMIENTO / "config.json"))
print("Variante  :", VARIANTE, CFG)
print("Bitácora  :", {k: cfg.get(k) for k in ["fecha", "gpu_nombre", "mascara"]}, "| lr", cfg["trainer"]["lr"],
      "| lote", cfg["trainer"]["batch_size"], "| semilla", cfg["seed"])
print("Elegido (huecos Arnold)     :", json.load(open(ec.DIR_ENTRENAMIENTO / "mejor.json")))
print("Elegido (huecos El Meslouhi):", json.load(open(RES_MESLOUHI / "mejor_huecos_meslouhi.json")))
print("Resultados:", GUARDADO / "resultados")